In [1]:
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import timedelta, datetime
from math import radians, sin, cos, sqrt, atan2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

def haversine_nm(lat1, lon1, lat2, lon2):
    R = 3440.065
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

def add_relative_direction_features(df):
    df['relative_wave_direction'] = (df['wave_direction'] - df['course']) % 360
    df['relative_wind_wave_direction'] = (df['wind_wave_direction'] - df['course']) % 360
    df['relative_swell_wave_direction'] = (df['swell_wave_direction'] - df['course']) % 360
    df['relative_wind_direction_10m'] = (df['wind_direction_10m'] - df['course'] + 360) % 360
    df['relative_ocean_current_direction'] = (df['ocean_current_direction'] - df['course'] + 360) % 360

    rel_angle_rad_wind_wave = np.deg2rad(df['relative_wind_wave_direction'])
    rel_angle_rad_swell_wave = np.deg2rad(df['relative_swell_wave_direction'])
    rel_angle_rad_current = np.deg2rad(df['relative_ocean_current_direction'])

    df['wind_wave_effect_forward'] = df['wind_wave_height'] * np.cos(rel_angle_rad_wind_wave)
    df['wind_wave_effect_side'] = df['wind_wave_height'] * np.sin(rel_angle_rad_wind_wave)

    df['swell_effect_forward'] = df['swell_wave_height'] * np.cos(rel_angle_rad_swell_wave)
    df['swell_effect_side'] = df['swell_wave_height'] * np.sin(rel_angle_rad_swell_wave)

    df['ocean_current_effect_forward'] = df['ocean_current_velocity'] * np.cos(rel_angle_rad_current)
    df['ocean_current_effect_side'] = df['ocean_current_velocity'] * np.sin(rel_angle_rad_current)

    df.drop(columns=['wave_direction', 'relative_wave_direction',
                     'wind_wave_direction', 'relative_wind_wave_direction',
                     'swell_wave_direction', 'relative_swell_wave_direction',
                     'wind_direction_10m', 'relative_wind_direction_10m',
                     'ocean_current_direction', 'relative_ocean_current_direction'], inplace=True)
    
    return df

/Users/itsgil/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
eta_df = pd.read_pickle("full_weather.pkl")
eta_df['date'] = pd.to_datetime(eta_df['date'])

vessel_info = pd.read_csv("vessel_info.csv")

# Merge vessel info
eta_df = eta_df.merge(
    vessel_info[['mmsi', 'speed_max']],
    on='mmsi',
    how='left'
)

# Delete irrelevant columns
eta_df = eta_df.drop(columns=['shipid', 'speed_kn', 'source', 'nav_status', 'draught', 'mmsi'])

eta_df

,journey_id,lat,lon,course,date,destination,destination_lat,destination_lon,wave_height,wave_direction,...,wind_wave_direction,wind_wave_period,swell_wave_height,swell_wave_direction,swell_wave_period,ocean_current_velocity,ocean_current_direction,wind_speed_10m,wind_direction_10m,speed_max
0,1,53.578350,8.552297,138.0,2022-03-14 17:26:00,None,NaN,NaN,0.14,285.0,...,275.0,1.65,0.04,321.0,9.00,0.2,90.0,13.0,265.0,20.0
1,1,53.578346,8.552303,100.0,2022-03-14 17:32:06,None,NaN,NaN,0.18,283.0,...,280.0,1.75,0.04,320.0,7.45,0.2,90.0,11.6,263.0,20.0
2,1,53.578346,8.552295,146.0,2022-03-14 17:53:00,None,NaN,NaN,0.18,283.0,...,280.0,1.75,0.04,320.0,7.45,0.2,90.0,11.6,263.0,20.0
3,1,53.578346,8.552292,317.0,2022-03-14 18:14:03,None,NaN,NaN,0.18,283.0,...,280.0,1.75,0.04,320.0,7.45,0.2,90.0,11.6,263.0,20.0
4,1,53.578346,8.552315,358.0,2022-03-14 18:26:00,None,NaN,NaN,0.18,283.0,...,280.0,1.75,0.04,320.0,7.45,0.2,90.0,11.6,263.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499428,396,31.128813,-81.520149,277.0,2023-10-11 20:36:19,None,NaN,NaN,0.20,98.0,...,15.0,0.25,0.18,86.0,9.65,0.5,315.0,9.3,54.0,20.0
499429,396,31.130751,-81.529221,292.0,2023-10-11 20:45:08,None,NaN,NaN,0.20,98.0,...,15.0,0.25,0.18,86.0,9.65,0.5,315.0,9.3,54.0,20.0
499430,396,31.134413,-81.537865,295.0,2023-10-11 20:57:00,None,NaN,NaN,0.20,98.0,...,15.0,0.25,0.18,86.0,9.65,0.5,315.0,9.3,54.0,20.0
499431,396,31.134333,-81.538116,238.0,2023-10-11 20:59:10,None,NaN,NaN,0.20,98.0,...,15.0,0.25,0.18,86.0,9.65,0.5,315.0,9.3,54.0,20.0


In [3]:
eta_df = add_relative_direction_features(eta_df)

eta_df['swell_impact'] = eta_df['swell_wave_height'] ** 2 * eta_df['swell_wave_period']
eta_df['wind_wave_energy'] = eta_df['wind_wave_height'] ** 2 * eta_df['wind_wave_period']

eta_df.drop(columns=['wave_height', 'wind_wave_height', 'swell_wave_height',
                     'wind_speed_10m','ocean_current_velocity',
                     'wave_period', 'swell_wave_period', 'wind_wave_period'], inplace=True)

eta_df.to_csv("course_deviation_storms_start_end.csv", index=False)
eta_df

,journey_id,lat,lon,course,date,destination,destination_lat,destination_lon,speed_max,wind_wave_effect_forward,wind_wave_effect_side,swell_effect_forward,swell_effect_side,ocean_current_effect_forward,ocean_current_effect_side,swell_impact,wind_wave_energy
0,1,53.578350,8.552297,138.0,2022-03-14 17:26:00,None,NaN,NaN,20.0,-0.087762,8.183980e-02,-0.039945,-0.002093,0.133826,-0.148629,0.01440,0.02376
1,1,53.578346,8.552303,100.0,2022-03-14 17:32:06,None,NaN,NaN,20.0,-0.160000,1.959435e-17,-0.030642,-0.025712,0.196962,-0.034730,0.01192,0.04480
2,1,53.578346,8.552295,146.0,2022-03-14 17:53:00,None,NaN,NaN,20.0,-0.111145,1.150944e-01,-0.039781,0.004181,0.111839,-0.165808,0.01192,0.04480
3,1,53.578346,8.552292,317.0,2022-03-14 18:14:03,None,NaN,NaN,20.0,0.127782,-9.629040e-02,0.039945,0.002093,-0.136400,0.146271,0.01192,0.04480
4,1,53.578346,8.552315,358.0,2022-03-14 18:26:00,None,NaN,NaN,20.0,0.033266,-1.565036e-01,0.031520,-0.024626,-0.006980,0.199878,0.01192,0.04480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499428,396,31.128813,-81.520149,277.0,2023-10-11 20:36:19,None,NaN,NaN,20.0,-0.000000,0.000000e+00,-0.176693,0.034346,0.394005,0.307831,0.31266,0.00000
499429,396,31.130751,-81.529221,292.0,2023-10-11 20:45:08,None,NaN,NaN,20.0,0.000000,0.000000e+00,-0.161783,0.078907,0.460252,0.195366,0.31266,0.00000
499430,396,31.134413,-81.537865,295.0,2023-10-11 20:57:00,None,NaN,NaN,20.0,0.000000,0.000000e+00,-0.157432,0.087266,0.469846,0.171010,0.31266,0.00000
499431,396,31.134333,-81.538116,238.0,2023-10-11 20:59:10,None,NaN,NaN,20.0,-0.000000,0.000000e+00,-0.158931,-0.084505,0.112476,0.487185,0.31266,0.00000


In [4]:
# Load and prep storm data
hurdat_df = pd.read_pickle("hurdat2_storm_data_2020_2025.pkl")
hurdat_df['storm_timestamp'] = pd.to_datetime(hurdat_df['datetime'])

# Parameters
D = 800  # nautical miles (strict for storm impact)
DELTA_HOURS = 48

D_HARD = 900  # outer ring for "hard negatives"
DELTA_HOURS_HARD = 48

labels = []  # will hold 1 (storm), -1 (hard negative), 0 (clean)
distances_to_storm = []
storm_winds = []

for idx, row in tqdm(eta_df.iterrows(), total=eta_df.shape[0], desc="Classifying entries"):
    lat, lon, date = row['lat'], row['lon'], pd.to_datetime(row['date'])

    # Step 1: Filter storms near in time (for both inner and outer zones)
    storms_time = hurdat_df[
        (hurdat_df['storm_timestamp'] >= date - pd.Timedelta(hours=DELTA_HOURS_HARD)) &
        (hurdat_df['storm_timestamp'] <= date + pd.Timedelta(hours=DELTA_HOURS_HARD))
    ].copy()

    # Step 2: Calculate distances
    storms_time['distance_nm'] = storms_time.apply(
        lambda x: haversine_nm(lat, lon, x['lat'], x['lon']), axis=1
    )

    # Step 3: Determine proximity
    min_dist = storms_time['distance_nm'].min() if not storms_time.empty else float('inf')
    if not storms_time.empty:
        # Find storm entries that are within strict threshold
        in_strict = storms_time[
            (storms_time['distance_nm'] <= D) &
            (storms_time['storm_timestamp'] >= date - pd.Timedelta(hours=DELTA_HOURS)) &
            (storms_time['storm_timestamp'] <= date + pd.Timedelta(hours=DELTA_HOURS))
        ]

        if not in_strict.empty:
            # Storm-affected: use closest storm in strict zone
            closest = in_strict.loc[in_strict['distance_nm'].idxmin()]
            labels.append(1)
            distances_to_storm.append(closest['distance_nm'])
            storm_winds.append(closest['max_sustained_wind'])
        elif min_dist <= D_HARD:
            # Hard negative: use closest storm in outer ring
            closest = storms_time.loc[storms_time['distance_nm'].idxmin()]
            labels.append(-1)
            distances_to_storm.append(closest['distance_nm'])
            storm_winds.append(closest['max_sustained_wind'])
        else:
            # Clean negative
            closest = storms_time.loc[storms_time['distance_nm'].idxmin()]
            labels.append(0)
            distances_to_storm.append(closest['distance_nm'])  # Closest storm distance
            storm_winds.append(closest['max_sustained_wind'])  # Closest storm intensity
    else:
        # No storms at all
        labels.append(0)
        distances_to_storm.append(99999)  # Large positive value for no storm
        storm_winds.append(0)  # No storm intensity

# Add to DataFrame
eta_df['storm_proximity_label'] = labels
eta_df['distance_to_storm_nm'] = distances_to_storm
eta_df['storm_wind'] = storm_winds
eta_df.to_csv("eta_with_storm_proximity_label.csv", index=False)

Classifying entries: 100%|██████████| 499433/499433 [04:07<00:00, 2021.95it/s]


In [5]:
TIME_WINDOW = 48  # hours

# Ensure date is in datetime format
eta_df['date'] = pd.to_datetime(eta_df['date'])

# Sort for efficient indexing
eta_df = eta_df.sort_values('date').reset_index(drop=True)

# Get timestamps of all storm-related entries (label 1 or -1)
storm_related = eta_df[eta_df['storm_proximity_label'].isin([1, -1])].copy()

# Create mask to drop only non-storm entries within ±48h of any storm-related entry
mask_to_drop = pd.Series(False, index=eta_df.index)

for ts in storm_related['date']:
    # Mark non-storm entries within window for dropping
    in_window = (
        (eta_df['date'] >= ts - pd.Timedelta(hours=TIME_WINDOW)) &
        (eta_df['date'] <= ts + pd.Timedelta(hours=TIME_WINDOW)) &
        (~eta_df['storm_proximity_label'].isin([1, -1]))
    )
    mask_to_drop |= in_window

# Keep all storm entries, and only non-storm entries outside the window
eta_df_clean = eta_df[~mask_to_drop].reset_index(drop=True)

# Final cleanup - DELETE PORT ENTRIES AS WELL
eta_df_clean = eta_df_clean.dropna()
eta_df_clean = eta_df_clean.drop(columns=['date'])
eta_df_clean

,journey_id,lat,lon,course,destination,destination_lat,destination_lon,speed_max,wind_wave_effect_forward,wind_wave_effect_side,swell_effect_forward,swell_effect_side,ocean_current_effect_forward,ocean_current_effect_side,swell_impact,wind_wave_energy,storm_proximity_label,distance_to_storm_nm,storm_wind
26941,194,50.271385,-1.388882,254.0,Brunswick,31.145,-81.493,21.0,0.564211,-6.490502e-01,1.829920,-0.192332,4.517930,1.295496,28.10048,2.51464,0,99999.000000,0
26943,194,50.250946,-1.508887,255.0,Brunswick,31.145,-81.493,21.0,0.578975,-7.149743e-01,1.861704,-0.261645,4.494632,1.374147,29.33552,2.96240,0,99999.000000,0
26945,194,50.164982,-1.978408,257.0,Brunswick,31.145,-81.493,21.0,0.576843,-9.600274e-01,1.955623,-0.309740,3.652794,1.047422,32.34330,4.76672,0,99999.000000,0
26946,194,50.151665,-2.058873,254.0,Brunswick,31.145,-81.493,21.0,0.631781,-9.728579e-01,1.989044,-0.209057,3.460542,0.992294,33.00000,5.18056,0,99999.000000,0
26947,194,50.107307,-2.323535,257.0,Brunswick,31.145,-81.493,21.0,0.639949,-1.154498e+00,1.969616,-0.347296,0.423935,0.678438,32.80000,7.23096,0,99999.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412747,263,31.129263,-81.519043,299.0,Brunswick,31.145,-81.493,20.0,0.013640,1.462707e-02,-0.190211,0.061803,0.288379,0.082691,0.22200,0.00030,-1,896.980728,35
412748,263,31.129593,-81.519379,9.0,Brunswick,31.145,-81.493,20.0,0.018410,-7.814623e-03,-0.006980,0.199878,0.176336,-0.242705,0.22200,0.00030,-1,896.986842,35
412749,263,31.129900,-81.518669,94.0,Brunswick,31.145,-81.493,20.0,-0.006180,-1.902113e-02,0.198509,0.024374,-0.226413,-0.196818,0.22200,0.00030,-1,896.945965,35
412750,263,31.129477,-81.518082,166.0,Brunswick,31.145,-81.493,20.0,-0.020000,2.449294e-18,0.084524,-0.181262,-0.257150,0.154511,0.22200,0.00030,-1,896.931010,35


In [ ]:
eta_df_clean['distance_to_destination_nm'] = eta_df_clean.apply(
    lambda row: haversine_nm(row['lat'], row['lon'], row['destination_lat'], row['destination_lon']),
    axis=1
)

# Drop other unneeded columns
eta_df_clean = eta_df_clean.drop(columns=['journey_id', 'destination', 'course', 'lat', 'lon', 'destination_lat', 'destination_lon'])

In [7]:
eta_df_clean["storm_proximity_label"].value_counts()
eta_df_clean.to_csv("eta_cleaned_sampling.csv", index=False)

In [44]:
eta_df_clean = pd.read_csv("eta_cleaned_sampling.csv")

HARD_NEGATIVES_SAMPLE_SIZE = 300
CLEAN_NEGATIVES_SAMPLE_SIZE = 6000

# hard negatives
hard_negatives = eta_df_clean[eta_df_clean['storm_proximity_label'] == -1]
hard_negatives = hard_negatives.dropna()

hard_neg_sample = hard_negatives.sample(n=HARD_NEGATIVES_SAMPLE_SIZE, random_state=41)

hard_neg_sample = hard_neg_sample.drop(columns=['storm_proximity_label'])

hard_neg_sample["target_delay_maneuver"] = 0
hard_neg_sample

# clean negatives
clean_negatives = eta_df_clean[eta_df_clean['storm_proximity_label'] == 0]
clean_negatives = clean_negatives.dropna()

clean_neg_sample = clean_negatives.sample(n=CLEAN_NEGATIVES_SAMPLE_SIZE, random_state=40)

clean_neg_sample = clean_neg_sample.drop(columns=['storm_proximity_label'])
clean_neg_sample["target_delay_maneuver"] = 0

# Concat samples
final_sample = pd.concat([hard_neg_sample, clean_neg_sample], ignore_index=True)
final_sample.to_csv("final_negative_samples.csv", index=False)